<a href="https://colab.research.google.com/github/SangwanJeon/sensex-corpus-workflow-public/blob/main/01_data_collection/pipelines/tiktok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install yt-dlp faster-whisper pympi-ling
!apt-get -qq update
!apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.1/182.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 81.0 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import os
from pathlib import Path

# ====== INPUT ======
TIKTOK_URL = "https://www.tiktok.com/@siamozeta/video/7610738008257482006?is_from_webapp=1&sender_device=pc" #here paste tiktok url

# two segments (start_sec, end_sec)
SEGMENTS = [
    (10.0, 65.0),   # Segment 1
    (40.0, 120.5),   # Segment 2
]

# Whisper
WHISPER_MODEL = "tiny"   # tiny, base, small, medium, large-v3
LANGUAGE = None          # "it" oppure None per autodetect

# Cartelle
WORKDIR = "/content/tiktok_work"
OUTDIR = os.path.join(WORKDIR, "outputs")

os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

print("WORKDIR:", WORKDIR)
print("OUTDIR :", OUTDIR)

WORKDIR: /content/tiktok_work
OUTDIR : /content/tiktok_work/outputs


In [ ]:
import subprocess, os

video_path = os.path.join(WORKDIR, "full.mp4")

cmd = [
    "yt-dlp",
    "-f", "bv*+ba/best",
    "--merge-output-format", "mp4",
    "-o", video_path,
    TIKTOK_URL
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("Saved:", video_path, "Size:", os.path.getsize(video_path))

Running: yt-dlp -f bv*+ba/best --merge-output-format mp4 -o /content/tiktok_work/full.mp4 https://www.tiktok.com/@siamozeta/video/7610738008257482006?is_from_webapp=1&sender_device=pc
Saved: /content/tiktok_work/full.mp4 Size: 15429293


In [ ]:
import subprocess, os

full_wav = os.path.join(WORKDIR, "full.wav")

cmd = [
    "ffmpeg", "-y",
    "-i", video_path,
    "-ac", "1",
    "-ar", "16000",
    "-vn",
    full_wav
]
subprocess.run(cmd, check=True)
print("Saved:", full_wav, "Size:", os.path.getsize(full_wav))

Saved: /content/tiktok_work/full.wav Size: 3963582


In [ ]:
import os, re
from faster_whisper import WhisperModel

use_cuda = os.path.exists("/proc/driver/nvidia")
device = "cuda" if use_cuda else "cpu"
compute_type = "float16" if use_cuda else "int8"

print("Device:", device, "Compute:", compute_type)

model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute_type)

segments_iter, info = model.transcribe(
    full_wav,
    language=LANGUAGE,
    vad_filter=True,
    word_timestamps=False
)

all_segs = []
for s in segments_iter:
    all_segs.append({
        "start": float(s.start),
        "end": float(s.end),
        "text": s.text.strip()
    })

print("Detected language:", info.language, "prob:", info.language_probability)
print("Transcribed segments:", len(all_segs))

# save  complete transcription
full_txt = os.path.join(WORKDIR, "full_transcript.txt")
with open(full_txt, "w", encoding="utf-8") as f:
    for s in all_segs:
        f.write(f"[{s['start']:.2f} - {s['end']:.2f}] {s['text']}\n")

print("Saved:", full_txt)

Device: cpu Compute: int8


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Detected language: it prob: 0.9809100031852722
Transcribed segments: 37
Saved: /content/tiktok_work/full_transcript.txt


In [ ]:
import re

def transcript_for_window(all_segs, start, end):
    picked = []
    for s in all_segs:
        if s["end"] <= start:
            continue
        if s["start"] >= end:
            continue
        picked.append(s["text"])
    txt = " ".join(picked)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

In [ ]:
import subprocess, os

segment_outputs = []

for i, (start, end) in enumerate(SEGMENTS, start=1):
    if end <= start:
        raise ValueError(f"Segmento {i} non valido: end <= start ({start}, {end})")

    seg_mp4 = os.path.join(OUTDIR, f"segment_{i:02d}.mp4")
    seg_wav = os.path.join(OUTDIR, f"segment_{i:02d}.wav")
    seg_txt = os.path.join(OUTDIR, f"segment_{i:02d}.txt")

    # MP4
    cmd_mp4 = [
        "ffmpeg", "-y",
        "-ss", str(start),
        "-to", str(end),
        "-i", video_path,
        "-c", "copy",
        seg_mp4
    ]
    res = subprocess.run(cmd_mp4, capture_output=True, text=True)
    if res.returncode != 0:
        cmd_mp4 = [
            "ffmpeg", "-y",
            "-ss", str(start),
            "-to", str(end),
            "-i", video_path,
            "-c:v", "libx264",
            "-c:a", "aac",
            "-movflags", "+faststart",
            seg_mp4
        ]
        subprocess.run(cmd_mp4, check=True)

    # WAV
    cmd_wav = [
        "ffmpeg", "-y",
        "-ss", str(start),
        "-to", str(end),
        "-i", full_wav,
        "-ac", "1",
        "-ar", "16000",
        seg_wav
    ]
    subprocess.run(cmd_wav, check=True)

    # TXT
    txt = transcript_for_window(all_segs, start, end)
    with open(seg_txt, "w", encoding="utf-8") as f:
        f.write(txt + "\n")

    segment_outputs.append({
        "index": i,
        "start": start,
        "end": end,
        "mp4": seg_mp4,
        "wav": seg_wav,
        "txt": seg_txt,
        "text": txt
    })

print("done: segments", len(segment_outputs))
segment_outputs

done: segments 2


[{'index': 1,
  'start': 10.0,
  'end': 65.0,
  'mp4': '/content/tiktok_work/outputs/segment_01.mp4',
  'wav': '/content/tiktok_work/outputs/segment_01.wav',
  'txt': '/content/tiktok_work/outputs/segment_01.txt',
  'text': "benvenuti a DOSTER WORLD, ma i cibbas, ho come dire in vero inolanda, i cibbas il posto doviti compro un pezzo di terra, il governo ti dice no, no davvero, fa il tuce, l'orometro do si chiama proprio fai da tu il posto che direvo è comprare il terreno, quanto grande, cosa costruirci sono praccone, tu e tu avvicini siete liberi di costruire il vostro quartiere e tutto di questo villaggio è realizzato il suo volene dei suoi habitanti e dai suoi habitanti, una sola è la regola, la agricoltura urbana è obligatoria, questo significa di almeno il 50% del poppezzamento deve essere destinato alla coltifazione del tuo cibo, contribuando così a un'oico sistema locale e sostenibile, e poi c'è le tradi diversi tipi di appezzamento. Tandard, il 58% di agricoltura, agricolo, lo 

In [ ]:
from pympi.Elan import Eaf
import os

eaf_paths = []

for seg in segment_outputs:
    eaf = Eaf()
    tier_name = "Segment"
    eaf.add_tier(tier_name)

    start_ms = 0
    end_ms = int(round((seg["end"] - seg["start"]) * 1000))
    value = seg["text"] if seg["text"] else "(no text)"

    # Nota: timeline locale (parte da 0 per ogni segmento)
    eaf.add_annotation(tier_name, start_ms, end_ms, value=value)

    eaf_path = os.path.join(OUTDIR, f"segment_{seg['index']:02d}.eaf")
    eaf.to_file(eaf_path)

    eaf_paths.append(eaf_path)
    print("Saved:", eaf_path)

eaf_paths

Saved: /content/tiktok_work/outputs/segment_01.eaf
Saved: /content/tiktok_work/outputs/segment_02.eaf


['/content/tiktok_work/outputs/segment_01.eaf',
 '/content/tiktok_work/outputs/segment_02.eaf']